In [1]:
import uuid
from langchain_chroma import Chroma
from langgraph.checkpoint.memory import InMemorySaver  
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever




c:\ProgramData\anaconda3\envs\FreeGenAI\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

In [3]:
import pickle
from pathlib import Path
from langchain_chroma import Chroma
from langchain_core.stores import InMemoryStore
from langchain_openai import OpenAIEmbeddings
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever

import os
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")


def load_retriever(load_path):
    """
    Load the retriever from saved location
    
    Args:
        load_path: Path to the folder containing saved retriever files
    
    Returns:
        MultiVectorRetriever: Loaded retriever ready to use
    """
    load_path = Path(load_path)
    
    # Define file paths
    chroma_path = load_path / "chroma_db"
    docstore_path = load_path / "docstore.pkl"
    metadata_path = load_path / "metadata.pkl"
    
    # Load metadata
    if metadata_path.exists():
        with open(metadata_path, 'rb') as f:
            metadata = pickle.load(f)
        print(f"✓ Metadata loaded")
        id_key = metadata['id_key']
        collection_name = metadata['collection_name']
    else:
        # Fallback values
        id_key = "doc_id"
        collection_name = "multi_modal_rag"
        print("⚠ Metadata not found, using defaults")
    
    # Load vectorstore
    print("Loading vectorstore...")
    vectorstore = Chroma(
        collection_name=collection_name,
        embedding_function=OpenAIEmbeddings(),
        persist_directory=str(chroma_path)
    )
    print(f"✓ Vectorstore loaded: {vectorstore._collection.count()} documents")
    
    # Load docstore
    print("Loading docstore...")
    store = InMemoryStore()
    if docstore_path.exists():
        with open(docstore_path, 'rb') as f:
            store.store = pickle.load(f)
        print(f"✓ Docstore loaded: {len(store.store)} parent documents")
    else:
        raise FileNotFoundError(f"Docstore not found at {docstore_path}")
    
    # Create retriever
    retriever = MultiVectorRetriever(
        vectorstore=vectorstore,
        docstore=store,
        id_key=id_key,
    )
    print("✅ Retriever loaded successfully!\n")
    
    return retriever

# Load the retriever
saved_path = r"C:\Users\dewan\Coding\MLOps\IAI_Solutions\saved_retriever"
retriever = load_retriever(saved_path)

# Now you can use it
results = retriever.invoke("your query here")
print(f"Found {len(results)} results")

TypeError: str expected, not NoneType

In [6]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI
from base64 import b64decode

import os
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

def parse_docs(docs):
    """Split base64-encoded images and texts"""
    b64 = []
    text = []
    for doc in docs:
        try:
            b64decode(doc)
            b64.append(doc)
        except Exception as e:
            text.append(doc)
    return {"images": b64, "texts": text}


def build_prompt(kwargs):

    docs_by_type = kwargs["context"]
    user_question = kwargs["question"]

    context_text = ""
    if len(docs_by_type["texts"]) > 0:
        for text_element in docs_by_type["texts"]:
            context_text += text_element.text

    # construct prompt with context (including images)
    prompt_template = f"""
    Answer the question based only on the following context, which can include text, tables, and the below image.
    Context: {context_text}
    Question: {user_question}
    """

    prompt_content = [{"type": "text", "text": prompt_template}]

    if len(docs_by_type["images"]) > 0:
        for image in docs_by_type["images"]:
            prompt_content.append(
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:image/jpeg;base64,{image}"},
                }
            )

    return ChatPromptTemplate.from_messages(
        [
            HumanMessage(content=prompt_content),
        ]
    )


chain = (
    {
        "context": retriever | RunnableLambda(parse_docs),
        "question": RunnablePassthrough(),
    }
    | RunnableLambda(build_prompt)
    | ChatOpenAI(model="gpt-4o-mini")
    | StrOutputParser()
)

chain_with_sources = {
    "context": retriever | RunnableLambda(parse_docs),
    "question": RunnablePassthrough(),
} | RunnablePassthrough().assign(
    response=(
        RunnableLambda(build_prompt)
        | ChatOpenAI(model="gpt-4o-mini")
        | StrOutputParser()
    )
)

TypeError: str expected, not NoneType

In [8]:
import pickle
from pathlib import Path
from langchain_chroma import Chroma
from langchain_core.stores import InMemoryStore
from langchain_openai import OpenAIEmbeddings
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever

import os
from dotenv import load_dotenv
load_dotenv()
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

# Define paths
persist_dir = r"C:\Users\dewan\Coding\MLOps\IAI_Solutions\saved_retriever\chroma_db"
docstore_path = Path(r"C:\Users\dewan\Coding\MLOps\IAI_Solutions\saved_retriever\docstore.pkl")
metadata_path = Path(r"C:\Users\dewan\Coding\MLOps\IAI_Solutions\saved_retriever\metadata.pkl")

# Initialize embeddings
embeddings = OpenAIEmbeddings()

# Load metadata
if metadata_path.exists():
    with open(metadata_path, 'rb') as f:
        metadata = pickle.load(f)
    id_key = metadata['id_key']
    collection_name = metadata['collection_name']
else:
    id_key = "doc_id"
    collection_name = "multi_modal_rag"

# Load the vectorstore
vectorstore = Chroma(
    collection_name=collection_name,
    embedding_function=embeddings,
    persist_directory=persist_dir
)
print(f"✓ Vectorstore loaded: {vectorstore._collection.count()} documents")

# Load the docstore
store = InMemoryStore()
if docstore_path.exists():
    with open(docstore_path, 'rb') as f:
        store.store = pickle.load(f)
    print(f"✓ Docstore loaded: {len(store.store)} parent documents")

# Recreate the retriever
retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    docstore=store,
    id_key=id_key,
)
print("✅ Retriever loaded successfully!")

# ============================================
# DIFFERENT WAYS TO USE THE RETRIEVER
# ============================================

# 1. Query with text (will be converted to vector automatically)
query_text = "What is the summary of the document?"
results = retriever.invoke(query_text)
print(f"\n1. Text query results: {len(results)} documents found")

# 2. Query with embedding vector directly
query_vector = embeddings.embed_query(query_text)
print(f"\n2. Query vector shape: {len(query_vector)} dimensions")

# Search vectorstore directly with vector
vector_results = vectorstore.similarity_search_by_vector(
    embedding=query_vector,
    k=5  # number of results
)
print(f"   Vector search results: {len(vector_results)} documents found")

# 3. Get similarity scores with vectors
results_with_scores = vectorstore.similarity_search_by_vector_with_relevance_scores(
    embedding=query_vector,
    k=5
)
print(f"\n3. Results with scores:")
for doc, score in results_with_scores:
    print(f"   Score: {score:.4f} - {doc.page_content[:100]}...")

# 4. Access the parent documents through retriever
# The retriever automatically fetches parent docs from docstore
parent_docs = retriever.invoke(query_text)
print(f"\n4. Parent documents retrieved: {len(parent_docs)}")
for i, doc in enumerate(parent_docs[:2], 1):
    print(f"\n   Parent Doc {i}:")
    print(f"   Content: {doc.page_content[:200]}...")
    print(f"   Metadata: {doc.metadata}")

# 5. Get embedding for a specific document
sample_text = "This is a sample text to embed"
embedding_vector = embeddings.embed_query(sample_text)
print(f"\n5. Embedding created: {len(embedding_vector)} dimensions")
print(f"   First 5 values: {embedding_vector[:5]}")

✓ Vectorstore loaded: 0 documents
✓ Docstore loaded: 182 parent documents
✅ Retriever loaded successfully!



1. Text query results: 0 documents found

2. Query vector shape: 1536 dimensions
   Vector search results: 0 documents found

3. Results with scores:

4. Parent documents retrieved: 0

5. Embedding created: 1536 dimensions
   First 5 values: [-0.020862514153122902, 0.00331444782204926, -0.007719129789620638, -0.005834774114191532, -0.0005316575989127159]


In [4]:
from langchain_chroma import Chroma
from langchain_core.stores import InMemoryStore
from langchain_openai import OpenAIEmbeddings
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
import pickle
import os
from dotenv import load_dotenv

load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# ---- LOAD VECTORSTORE ----
vectorstore = Chroma(
    collection_name="multi_modal_rag",
    embedding_function=OpenAIEmbeddings(),
    persist_directory="./saved_retriever2/chroma_db"
)

# ---- LOAD DOCSTORE ----
with open("./saved_retriever/docstore.pkl", "rb") as f:
    docstore_data = pickle.load(f)

# Put into InMemoryStore
docstore = InMemoryStore()
items = list(docstore_data.items())
docstore.mset(items)


# ---- RESTORE RETRIEVER ----
retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    docstore=docstore,
    id_key="doc_id"
)

print("Retriever loaded!")


Retriever loaded!
